# 6 — Agent (Google ADK)

**AWS-side equivalent:** Strands Agents — `notebooks/aws/06-strands-agent.ipynb` (Agent + @tool + BedrockModel). 
**GCP service:** [Google Agent Development Kit](https://google.github.io/adk-docs/) — `google.adk.agents.Agent` (alias for `LlmAgent`), tools are plain Python callables (the framework introspects type hints + docstrings), `Runner` orchestrates the reason-act loop.

Package: `google-adk` (pinned in `notebooks/gcp/requirements.txt`). Python ≥ 3.11 required. 

This notebook mirrors the Strands surface: define `search_organization` + `get_expertise_scores` tools that hit AlloyDB (or Cloud SQL PG) via the same SQL the AWS notebook executes against Aurora, then runs the agent locally with the in-memory session service.

## Service surface (as of 2026-05-20)

- `from google.adk.agents import Agent` (alias of `LlmAgent`) — tools are passed as a list of plain `def`/`async def` callables; there is no decorator like Strands' `@tool`. Type hints + docstring become the tool schema.
- `from google.adk.runners import Runner` — `runner.run_async(user_id=..., session_id=..., new_message=Content(...))` yields `Event` objects.
- `from google.adk.sessions import InMemorySessionService` for local dev. Replace with `VertexAiSessionService` against an Agent Engine when deploying.
- Model strings are bare Gemini IDs (e.g. `gemini-2.5-flash`); Vertex backend resolves automatically when `GOOGLE_GENAI_USE_VERTEXAI=TRUE` or the constructor is given Vertex config.

Docs: <https://google.github.io/adk-docs/agents/llm-agents/>, <https://google.github.io/adk-docs/tools/function-tools/>

## Step 1: Load config + point ADK at Vertex

In [ ]:
import json
import os
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

GEMINI_MODEL = gcp.get('text_model', 'gemini-2.5-flash')

# ADK / google-genai both honor these env vars to route to Vertex AI (instead of the public Gemini API).
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION

print(f'project={PROJECT_ID} region={REGION} model={GEMINI_MODEL}')

## Step 2: Define the org-graph tools

These match the surface of the Strands tools in `notebooks/aws/06-strands-agent.ipynb` but read from AlloyDB / Cloud SQL PG instead of Aurora.

Notebook 02 (`02-alloydb-graph.ipynb`) is expected to write `gcp.alloydb.instance_uri` (or `gcp.cloudsql.dsn`) into `config.json`. We fall back to an in-memory stub if no DB is wired yet, so the agent loop still runs end-to-end.

In [ ]:
from typing import Any

DSN = gcp.get('alloydb_dsn') or gcp.get('cloudsql_dsn')

_STUB_PEOPLE = [
    {'name': 'Alice Chen', 'title': 'Staff Platform Eng', 'skills': [('Kubernetes', 0.95), ('GitOps', 0.8)]},
    {'name': 'Bob Kumar',  'title': 'Staff ML Engineer', 'skills': [('PyTorch', 0.92), ('Vertex AI', 0.88), ('MLOps', 0.9)]},
    {'name': 'Dave Park',  'title': 'Sr Platform Eng',  'skills': [('Terraform', 0.9), ('CI/CD', 0.93), ('Kubernetes', 0.7)]},
    {'name': 'Eve Park',   'title': 'Security Lead',     'skills': [('Zero Trust', 0.95), ('IAM', 0.9)]},
]

def _connect():
    if not DSN:
        return None
    import psycopg
    return psycopg.connect(DSN)

def search_organization(skill: str, limit: int = 5) -> dict[str, Any]:
    """Find people in the org with expertise in a given skill or domain.

    Args:
        skill: skill or domain to search for (e.g. "Kubernetes", "ML", "security").
        limit: maximum people to return.

    Returns:
        dict with key 'results': list of {name, title, skill, strength}.
    """
    conn = _connect()
    if conn is None:
        rows = []
        for p in _STUB_PEOPLE:
            for s, w in p['skills']:
                if skill.lower() in s.lower():
                    rows.append({'name': p['name'], 'title': p['title'], 'skill': s, 'strength': w})
        rows.sort(key=lambda r: -r['strength'])
        return {'results': rows[:limit], 'source': 'stub'}
    try:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT p.name, p.title, s.name AS skill, COALESCE(hs.weight, 1.0) AS strength
                FROM people p
                JOIN has_skill hs ON hs.person_id = p.id
                JOIN skills s     ON s.id        = hs.skill_id
                WHERE lower(s.name) LIKE %s
                ORDER BY strength DESC
                LIMIT %s
                ''',
                (f'%{skill.lower()}%', limit),
            )
            rows = [{'name': n, 'title': t, 'skill': s, 'strength': float(w)} for n, t, s, w in cur.fetchall()]
        return {'results': rows, 'source': 'alloydb'}
    finally:
        conn.close()

def get_expertise_scores(person_name: str) -> dict[str, Any]:
    """Return composite expertise scores for a named person across their top skills.

    Args:
        person_name: full or partial name (case-insensitive).

    Returns:
        dict with key 'scores': list of {skill, strength}, plus 'person' metadata.
    """
    conn = _connect()
    if conn is None:
        for p in _STUB_PEOPLE:
            if person_name.lower() in p['name'].lower():
                return {
                    'person': {'name': p['name'], 'title': p['title']},
                    'scores': [{'skill': s, 'strength': w} for s, w in p['skills']],
                    'source': 'stub',
                }
        return {'person': None, 'scores': [], 'source': 'stub'}
    try:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT p.id, p.name, p.title, s.name, COALESCE(hs.weight, 1.0)
                FROM people p
                LEFT JOIN has_skill hs ON hs.person_id = p.id
                LEFT JOIN skills s     ON s.id        = hs.skill_id
                WHERE lower(p.name) LIKE %s
                ORDER BY hs.weight DESC NULLS LAST
                LIMIT 25
                ''',
                (f'%{person_name.lower()}%',),
            )
            rows = cur.fetchall()
        if not rows:
            return {'person': None, 'scores': []}
        _, name, title, _, _ = rows[0]
        return {
            'person': {'name': name, 'title': title},
            'scores': [{'skill': s, 'strength': float(w)} for _, _, _, s, w in rows if s],
            'source': 'alloydb',
        }
    finally:
        conn.close()

print('search_organization("Kubernetes") →', search_organization('Kubernetes', limit=3))
print('get_expertise_scores("Bob") →', get_expertise_scores('Bob'))

## Step 3: Build the ADK Agent

The system prompt is the GCP twin of the Strands `SYSTEM_PROMPT` in the AWS notebook.

In [ ]:
from google.adk.agents import Agent
from google.genai import types

SYSTEM_PROMPT = (
    'You are OrgIQ Search, an organizational intelligence assistant.\n'
    'You help users find experts within the organization.\n\n'
    'You have these tools:\n'
    '- search_organization(skill, limit) — find people with a given skill\n'
    '- get_expertise_scores(person_name) — pull a person\'s top skills\n\n'
    'Always cite the people by name in your answer. Prefer concise bullet points.\n'
)

agent = Agent(
    model=GEMINI_MODEL,
    name='orgiq_search_agent',
    description='Org expertise lookup across people, titles, and skills.',
    instruction=SYSTEM_PROMPT,
    tools=[search_organization, get_expertise_scores],
    generate_content_config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=512),
)

print(f'✓ ADK Agent ready (model={GEMINI_MODEL}, tools={[t.__name__ for t in [search_organization, get_expertise_scores]]})')

## Step 4: Run the agent locally and dump the tool-call trace

`Runner.run_async` is an async generator of `Event`s — each event can carry partial text, a tool call, a tool result, or a final response. We collect everything so the trace can be compared to the Strands tool-call log.

In [ ]:
import asyncio
import uuid
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

APP_NAME = 'orgiq-notebook'
USER_ID = 'user-notebook'

session_service = InMemorySessionService()
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

async def ask(question: str) -> dict:
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=str(uuid.uuid4()))
    new_message = Content(role='user', parts=[Part(text=question)])
    trace = []
    final_text = ''
    async for event in runner.run_async(user_id=USER_ID, session_id=session.id, new_message=new_message):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, 'function_call', None):
                    fc = part.function_call
                    trace.append({'kind': 'tool_call', 'name': fc.name, 'args': dict(fc.args or {})})
                if getattr(part, 'function_response', None):
                    fr = part.function_response
                    trace.append({'kind': 'tool_result', 'name': fr.name, 'response': dict(fr.response or {})})
                if getattr(part, 'text', None) and event.is_final_response():
                    final_text += part.text
    return {'final_text': final_text, 'trace': trace, 'session_id': session.id}

result = await ask('Who knows Kubernetes best, and what other strong skills do they have?')
print('FINAL:\n', result['final_text'])
print('\nTRACE:')
for step in result['trace']:
    print(f'  - {step["kind"]:11s} {step["name"]}: {json.dumps(step.get("args") or step.get("response"))[:160]}')

## Step 5: Multi-turn — reuse the same session

In [ ]:
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=str(uuid.uuid4()))

async def turn(text: str) -> str:
    new_message = Content(role='user', parts=[Part(text=text)])
    out = ''
    async for event in runner.run_async(user_id=USER_ID, session_id=session.id, new_message=new_message):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if getattr(part, 'text', None):
                    out += part.text
    return out

print('Turn 1:\n', await turn('Who is our strongest MLOps person?'))
print('\nTurn 2:\n', await turn('What other skills do they have?'))

## Step 6: Save state

Downstream notebooks (`07-agent-engine-memory`, `15-deploy-agent-engine`) will reuse this agent definition. We record the model + system-prompt fingerprint in `config.json` so the deploy notebook can rebuild the same agent.

In [ ]:
gcp['adk_agent'] = {
    'name': agent.name,
    'model': GEMINI_MODEL,
    'tools': [search_organization.__name__, get_expertise_scores.__name__],
    'instruction_preview': SYSTEM_PROMPT[:120],
}
Path('config.json').write_text(json.dumps(config, indent=2))
print('✓ config.json updated with gcp.adk_agent')

## Cross-reference vs AWS

| AWS step (`06-strands-agent.ipynb`)           | GCP equivalent here                                 |
|------------------------------------------------|-----------------------------------------------------|
| `@tool def search_experts(...)`               | plain `def search_organization(...)` — type hints + docstring become schema |
| `BedrockModel(model_id=...)`                  | model passed as bare string to `Agent(model=...)`   |
| `Agent(model, system_prompt, tools)`          | `Agent(model, instruction, tools, description)`     |
| `agent("Who knows Kubernetes?")`              | `Runner.run_async(... new_message=Content(...))`    |
| Multi-turn via re-using `agent`               | Multi-turn via re-using `session_id` in the runner  |

**Next:** [07-agent-engine-memory.ipynb](07-agent-engine-memory.ipynb) — add persistent memory via Agent Engine Memory Bank.